# Week 8: Shine Your Eye – Nigerian Multi-Agent (Prod-Ready)

**PR:** Prod-ready multi-agent architecture. Nigerian-relatable: Naira rates, local deals, and FAQs (NIN, Lagos, etc.).

- **Router Agent** – classifies intent (naira_rate | deals | faq | general).
- **Naira Agent** – Naira/USD rate + CBN-style info (structured Pydantic).
- **Deal Scanner Agent** – Nigerian-style deals (Jumia/Konga); structured outputs.
- **RAG Agent** – ChromaDB over Nigerian FAQs; answers from vector store.
- **Orchestrator** – runs router, dispatches to agents, returns formatted response.
- **Gradio UI** – ask in plain English or Pidgin; see agent flow + answer.

Run cells in order. Set `OPENROUTER_API_KEY` in `.env`.  
**Telegram:** Optional. Create a bot via [@BotFather](https://t.me/BotFather), get the token, then set `TELEGRAM_BOT_TOKEN` and `TELEGRAM_CHAT_ID` in `.env`. Check "Send report to Telegram" in the UI to get the answer in your chat.  
**Prod-ready:** typed agents, Pydantic schemas, logging, RAG (ChromaDB). For serverless, deploy any agent to Modal and call from the orchestrator.

In [ ]:
import os
import json
import logging
import requests
import numpy as np
from typing import Optional, List
from pydantic import BaseModel, Field
from openai import OpenAI
from dotenv import load_dotenv
import chromadb
from chromadb.config import Settings
import gradio as gr
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import plotly.graph_objects as go

load_dotenv()
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
TELEGRAM_BOT_TOKEN = os.getenv("TELEGRAM_BOT_TOKEN")
TELEGRAM_CHAT_ID = os.getenv("TELEGRAM_CHAT_ID")
BASE_URL = "https://openrouter.ai/api/v1"
MODEL = "openai/gpt-4o-mini"

In [ ]:
logging.basicConfig(level=logging.INFO, format="[%(asctime)s] %(message)s")
logger = logging.getLogger(__name__)

class Agent:
    name = ""
    color = "\033[37m"
    RED, GREEN, YELLOW, BLUE, CYAN, RESET = "\033[31m", "\033[32m", "\033[33m", "\033[34m", "\033[36m", "\033[0m"

    def log(self, msg: str):
        logging.info(f"{self.color}[{self.name}] {msg}{self.RESET}")

In [ ]:
class RouterIntent(BaseModel):
    intent: str = Field(description="One of: naira_rate, deals, faq, general")
    reason: str = Field(description="Brief reason for this intent")

class NairaRate(BaseModel):
    rate_usd_ngn: float = Field(description="Naira per 1 USD")
    source: str = Field(description="e.g. CBN, parallel market")
    note: str = Field(description="Short note")

class NigerianDeal(BaseModel):
    title: str = Field(description="Product title")
    price_ngn: float = Field(description="Price in Naira")
    category: str = Field(description="e.g. Phones, Fashion")
    url: str = Field(description="Link to deal")

class DealList(BaseModel):
    deals: List[NigerianDeal] = Field(description="Up to 5 Nigerian-style deals")

In [ ]:
def llm_json(client: OpenAI, system: str, user: str, response_format: Optional[dict] = None):
    kwargs = {"model": MODEL, "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}]}
    if response_format:
        kwargs["response_format"] = {"type": "json_schema", "json_schema": {"name": "r", "strict": True, "schema": response_format}}
    r = client.chat.completions.create(**kwargs)
    return r.choices[0].message.content

In [ ]:
class RouterAgent(Agent):
    name = "Router"
    color = Agent.GREEN

    def route(self, query: str, client: OpenAI) -> RouterIntent:
        self.log(f"Routing: {query[:60]}...")
        system = "You classify user intent. Reply with JSON: {intent: one of naira_rate, deals, faq, general, reason: short reason}."
        user = f"User query: {query}"
        raw = llm_json(client, system, user)
        try:
            d = json.loads(raw)
            out = RouterIntent(intent=d.get("intent", "general"), reason=d.get("reason", ""))
        except Exception:
            out = RouterIntent(intent="general", reason="parse failed")
        self.log(f"Intent: {out.intent}")
        return out

In [ ]:
class NairaAgent(Agent):
    name = "Naira"
    color = Agent.CYAN

    def get_rate(self, client: OpenAI) -> NairaRate:
        self.log("Fetching Naira/USD rate (simulated).")
        return NairaRate(rate_usd_ngn=1580.0, source="CBN (simulated)", note="Official window. Parallel market often higher.")

    def run(self, query: str, client: OpenAI) -> str:
        rate = self.get_rate(client)
        return f"1 USD = ₦{rate.rate_usd_ngn:,.2f} ({rate.source}). {rate.note}"

In [ ]:
class DealScannerAgent(Agent):
    name = "DealScanner"
    color = Agent.YELLOW

    MOCK_DEALS = [
        NigerianDeal(title="Samsung A54 5G", price_ngn=285000, category="Phones", url="https://jumia.com.ng/example1"),
        NigerianDeal(title="Nivea Body Lotion 400ml", price_ngn=4500, category="Beauty", url="https://konga.com/example2"),
        NigerianDeal(title="Nigerian Jollof Rice Spice Pack", price_ngn=2500, category="Food", url="https://jumia.com.ng/example3"),
    ]

    def run(self, query: str, client: OpenAI) -> str:
        self.log("Scanning Nigerian deals (mock).")
        lines = [f"• {d.title} – ₦{d.price_ngn:,.0f} ({d.category})" for d in self.MOCK_DEALS]
        return "Deals today:\n" + "\n".join(lines)

In [ ]:
NIGERIAN_FAQS = [
    "NIN is National Identification Number. You can enroll at NIMC offices or approved centers. Bring BVN if you have it.",
    "Lagos has 20 LGAs. Popular areas: Ikeja, Lekki, Victoria Island, Surulere, Yaba, Ajah.",
    "CBN sets official Naira rate. Parallel (black) market rate is often higher. Check CBN website for official.",
    "BVN links your bank accounts. Get it at any bank. You need valid ID and biometrics.",
]

def build_rag():
    client = chromadb.Client(Settings(anonymized_telemetry=False))
    coll = client.get_or_create_collection("naija_faq", metadata={"description": "Nigerian FAQs"})
    if coll.count() == 0:
        for i, text in enumerate(NIGERIAN_FAQS):
            coll.add(ids=[str(i)], documents=[text], metadatas=[{"id": i}])
    return coll

rag_collection = build_rag()

In [ ]:
class RAGAgent(Agent):
    name = "RAG"
    color = Agent.BLUE

    def __init__(self, collection):
        self.coll = collection

    def run(self, query: str, client: OpenAI) -> str:
        self.log("Querying Nigerian FAQ RAG.")
        res = self.coll.query(query_texts=[query], n_results=2)
        docs = res["documents"][0] if res["documents"] else []
        if not docs:
            return "No matching FAQ. Try asking about NIN, Lagos, CBN, or BVN."
        return "\n".join(docs)

In [ ]:
# 3D visualization of RAG (ChromaDB) embeddings – t-SNE or PCA
FAQ_LABELS = ["NIN", "Lagos", "CBN", "BVN"]
COLORS = ["#9b59b6", "#3498db", "#e67e22", "#2ecc71"]

def get_rag_viz_plot(collection):
    """Build 3D scatter of FAQ embeddings (t-SNE/PCA). Returns Plotly figure for Gradio."""
    data = collection.get(include=["embeddings", "documents", "metadatas"])
    if not data["ids"]:
        fig = go.Figure()
        fig.add_annotation(text="No RAG data yet.", x=0.5, y=0.5, showarrow=False)
        fig.update_layout(title="RAG Embedding Space (empty)")
        return fig
    emb = np.array(data["embeddings"], dtype=np.float64)
    n = len(emb)
    if n <= 3:
        coords = PCA(n_components=min(3, emb.shape[1])).fit_transform(emb)
    else:
        coords = TSNE(n_components=3, random_state=42, perplexity=min(5, n - 1)).fit_transform(emb)
    colors = [COLORS[i % len(COLORS)] for i in range(n)]
    labels = [FAQ_LABELS[i] if i < len(FAQ_LABELS) else f"Doc {i}" for i in range(n)]
    fig = go.Figure(data=[
        go.Scatter3d(
            x=coords[:, 0], y=coords[:, 1], z=coords[:, 2],
            mode="markers+text",
            marker=dict(size=12, color=colors, opacity=0.9),
            text=labels, textposition="top center",
        )
    ])
    fig.update_layout(
        title="RAG FAQ Embedding Space (t-SNE 3D)",
        scene=dict(xaxis_title="X", yaxis_title="Y", zaxis_title="Z", bgcolor="#1e1e2e"),
        paper_bgcolor="#1e1e2e", font=dict(color="#eee"), height=450,
    )
    return fig

In [ ]:
class Orchestrator:
    def __init__(self):
        self.client = OpenAI(api_key=OPENROUTER_API_KEY, base_url=BASE_URL) if OPENROUTER_API_KEY else None
        self.router = RouterAgent()
        self.naira = NairaAgent()
        self.deals = DealScannerAgent()
        self.rag = RAGAgent(rag_collection)

    def run(self, query: str) -> tuple:
        """Returns (flow_md, answer) for Gradio visualization."""
        steps = []
        if not self.client:
            return "**Flow:** (no API key)\n", "Set OPENROUTER_API_KEY in .env to use agents."
        steps.append(f"📥 **Input:** {query[:80]}...")
        intent = self.router.route(query, self.client)
        steps.append(f"🟢 **Router** → intent: `{intent.intent}` ({intent.reason})")
        if intent.intent == "naira_rate":
            steps.append("🔵 **Naira** → fetching rate...")
            out = self.naira.run(query, self.client)
            steps.append("🔵 **Naira** → done")
        elif intent.intent == "deals":
            steps.append("🟡 **DealScanner** → scanning...")
            out = self.deals.run(query, self.client)
            steps.append("🟡 **DealScanner** → done")
        elif intent.intent == "faq":
            steps.append("🔷 **RAG** → querying FAQ...")
            out = self.rag.run(query, self.client)
            steps.append("🔷 **RAG** → done")
        else:
            out = f"Intent '{intent.intent}' – try: naira rate, deals, or NIN/Lagos/CBN/BVN."
        flow_md = "**Agent flow**\n\n" + "\n".join(steps)
        return flow_md, out

In [ ]:
# Quick test (no Gradio): run orchestrator once
# print(chat("How much is dollar to naira?"))
# print(chat("Wetin be NIN?"))

In [ ]:
def send_report_to_telegram(query: str, flow_md: str, answer: str) -> str:
    """Send a short report to Telegram. Returns status message."""
    if not TELEGRAM_BOT_TOKEN or not TELEGRAM_CHAT_ID:
        return "Telegram not configured (set TELEGRAM_BOT_TOKEN and TELEGRAM_CHAT_ID in .env)."
    text = (
        f"🇳🇬 *Shine Your Eye – Report*\n\n"
        f"*Query:* {query[:200]}\n\n"
        f"*Answer:*\n{answer[:3500]}"
    )
    url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendMessage"
    try:
        r = requests.post(url, json={"chat_id": TELEGRAM_CHAT_ID, "text": text, "parse_mode": "Markdown"}, timeout=10)
        if r.ok:
            return "✅ Report sent to Telegram."
        return f"❌ Telegram error: {r.status_code} {r.text[:100]}"
    except Exception as e:
        return f"❌ Telegram error: {e}"

In [ ]:
_orch = None
def get_orch():
    global _orch
    if _orch is None:
        _orch = Orchestrator()
    return _orch

def chat(query: str, send_to_telegram: bool):
    flow_md, answer = get_orch().run(query)
    telegram_status = ""
    if send_to_telegram:
        telegram_status = send_report_to_telegram(query, flow_md, answer)
    return flow_md, answer, telegram_status

with gr.Blocks(title="Shine Your Eye", theme=gr.themes.Soft()) as app:
    gr.Markdown("# 🇳🇬 Shine Your Eye – Nigerian Multi-Agent")
    with gr.Tabs():
        with gr.TabItem("Chat"):
            gr.Markdown("Ask in English or Pidgin. **Agent flow** shows which agents ran.")
            with gr.Row():
                inp = gr.Textbox(placeholder="e.g. How much is dollar to naira?", label="Ask", scale=2)
                send_tg = gr.Checkbox(label="📱 Send to Telegram", value=False)
                btn = gr.Button("Go", variant="primary")
            with gr.Row():
                flow_out = gr.Markdown(label="Agent flow", value="*Agent steps will appear here.*")
                ans_out = gr.Textbox(label="Answer", lines=8, interactive=False)
            tg_status = gr.Textbox(label="Telegram", value="", interactive=False)
            def run_chat(q, stg):
                flow_md, answer, status = chat(q, stg)
                return flow_md, answer, status
            btn.click(fn=run_chat, inputs=[inp, send_tg], outputs=[flow_out, ans_out, tg_status])
            inp.submit(fn=run_chat, inputs=[inp, send_tg], outputs=[flow_out, ans_out, tg_status])
        with gr.TabItem("RAG embedding viz"):
            gr.Markdown("3D view of FAQ embeddings (t-SNE). Each point = one RAG document.")
            viz_btn = gr.Button("Show 3D embedding plot", variant="secondary")
            viz_plot = gr.Plot(label="RAG vector space")
            viz_btn.click(fn=lambda: get_rag_viz_plot(rag_collection), inputs=None, outputs=viz_plot)
app.launch(inbrowser=False)